# Lab: Mapping Solar Panel Output with Contour Maps

**Intro to Data Science**

## Your role

You are part of a **renewable energy engineering team**. Your team maintains
a fleet of solar panels and is investigating how much output is lost to
everyday soiling (dust, pollen, debris) between cleanings. Facilities
management has a limited maintenance budget and wants your team's
recommendation on **where to focus cleaning effort** -- not just a
description of "how dirty" the panel is.

To answer that, your team scanned a test panel with a single light source in
a grid pattern, pausing at each point to record the panel's output voltage.
This was done twice: once with the panel **clean**, once **dirty**. Your job
is to turn those two raw sensor logs into three **contour maps** -- clean,
dirty, and the difference between them -- and use the evidence to answer your
team's guiding question:

> **Where on the panel is soiling costing the most output, and how does it
> affect the panel's efficiency?**

By the end of the lab, you'll use that evidence to make an actual
maintenance recommendation under a real constraint (Section 10).

## Learning objectives

By the end of this lab you will be able to:

**Computational / data science:**
- Load and inspect a real, messy, instrument-exported data file (metadata header + raw samples)
- Slice a continuous time-series signal into repeated measurement windows and
  aggregate each one
- Reshape 1D scan data into a 2D grid, accounting for a serpentine (zig-zag)
  collection order
- Build and interpret **contour maps**, including a **diverging** colormap for
  a signed difference

**Engineering / scientific reasoning:**
- Interpret experimental data to identify where and how a physical system is
  underperforming
- Connect patterns in a visualization back to a plausible physical cause
- Use quantitative evidence to justify an engineering recommendation made
  under a real-world constraint (limited budget, limited time)

**Grading:** every question and coding exercise is worth points, noted where
it appears. Written/interpretation questions total 65 points and coding
exercises total 35 points, for **100 points** overall. (The warm-up in
Section 0 is not graded.)

## How the scan works

- The light starts at the **top-right** corner of the panel.
- **Row 1**: moves **right -> left**, pausing 3 seconds at each of 11 points.
- Drops down one row. **Row 2**: moves **left -> right**, 11 points.
- Repeats, zig-zagging (this is called a *serpentine* or *boustrophedon* scan),
  for 22 rows total -> **11 x 22 = 242 points**.
- At each point: **3s dwell** (light is stationary -- this is what we measure)
  + **3s move** (light is travelling to the next point -- this is discarded).

## About the data file

The measurement rig logs a continuous voltage trace to a `.csv` file with a
metadata header block (recording start/end time, sample rate, channel name,
etc.), a blank line, and then two columns with no header row: `time_ms, volts`,
sampled every 10ms. A parser (provided for you in `solar_helpers.py`) pulls the sample rate out of the
header and finds where the numeric data actually starts.

## 0. Warm-up: what do you notice, what do you wonder? (not graded)

*If your instructor is showing photos or a short video of the actual clean
and dirty test panel, take a look now before continuing.*

Below are contour maps from two scans of the same panel -- **Panel A** and
**Panel B** -- with the color scale intentionally unlabeled. One of them is
the clean scan; the other is the dirty scan. Don't do any calculations yet --
just look.

![Panel A](output/mystery_panel_A.png)  ![Panel B](output/mystery_panel_B.png)

**Discuss with a partner, then jot down brief answers (a sentence or two each):**

1. **What do you notice?** How would you describe the pattern in each map?
2. **What do you wonder?** What questions do these patterns raise about the panel or the scan?
3. **Predict:** which one do you think is the *dirty* scan, and specifically
   *where* on that panel do you predict the biggest performance loss will
   turn out to be? What's your reasoning?

You'll find out if you were right once you build the real maps yourself.

**Your answers:**

> _(replace this)_

## 1. Setup

This lab uses a helper file, `solar_helpers.py`, that must sit in the **same folder** as this notebook. It holds the longer pieces of code (reading the data file, averaging each measurement window, and drawing the contour maps) so you can focus on the data. You're welcome to open it and look, but you don't need to change it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Helper functions provided for you (see solar_helpers.py)
from solar_helpers import parse_scope_csv, extract_point_values, plot_panel_contour

## 2. Configuration

In [ ]:
# ---- Files ----
CSV_PATH_CLEAN = "data/lab_clean_r1_test_3s_interval.csv"
CSV_PATH_DIRTY = "data/lab_dirty_r1_test_3s_interval.csv"

# Where to save the output plots (PNG files)
OUTPUT_DIR = "output"

# ---- Scan timing (seconds) ----
DWELL_S = 3.0          # time light sits still at each point (this is what we measure)
MOVE_S = 3.0           # time given to move to the next point (discarded)
TRIM_S = 0.5           # seconds trimmed off the START of each dwell window
                       # (lets the reading settle after the light stops moving)
MIN_WINDOW_FRAC = 0.25 # if the recording ends early, the last window is still used
                       # as long as at least this fraction of a full window is available

# ---- Grid geometry ----
N_X = 11               # points across the width (left-right)
N_Y = 22               # points down the length (top-bottom)

# ---- Physical panel size (cm) ----
# Measured as 17in x 37.5in, converted to cm (x 2.54).
PANEL_WIDTH_CM = 17.0 * 2.54
PANEL_LENGTH_CM = 37.5 * 2.54

# ---- Scan start / direction ----
# Starts top-right. Row 0 moves right -> left. Row 1 moves left -> right. Etc.
START_TOP_RIGHT = True

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Parse the raw CSVs

The scope-export CSV has a metadata header of variable length, then a blank line,
then two columns: `time_ms, volts` (no header row). We auto-detect where the
data actually starts, and pull the logger's sample interval (`SampleTime[ms]`)
straight out of the header. The code that does this lives in `solar_helpers.py`; the cell below just calls it on both files.

In [ ]:
df_clean, sample_time_clean = parse_scope_csv(CSV_PATH_CLEAN)
df_dirty, sample_time_dirty = parse_scope_csv(CSV_PATH_DIRTY)

print(f"Clean: {len(df_clean):,} samples, {sample_time_clean} ms/sample, "
      f"{df_clean['time_s'].iloc[-1]:.2f}s total")
print(f"Dirty: {len(df_dirty):,} samples, {sample_time_dirty} ms/sample, "
      f"{df_dirty['time_s'].iloc[-1]:.2f}s total")

if sample_time_clean != sample_time_dirty:
    print("WARNING: clean and dirty files have different sample rates -- "
          "double check these are comparable recordings.")

**Comprehension check (4 points):** why can't we just assume the data
always starts on, say, line 25? What about this file format makes that unsafe?

**Your answer:**

> _(replace this)_

### 3a. Look before you leap: inspect and plot the raw trace

Before doing anything else, always look at your raw data. Run the first cell below
to see what `df_clean` actually contains: one row per reading, with a
timestamp and the panel's output voltage. Notice how many rows there are and
how closely spaced they are.

Then plot the signal itself.

**Code exercise (3 points):** plot the first ~30 seconds of `df_clean`'s
voltage trace against time (that's about five scan points, since each point
takes 6 seconds: 3s dwell + 3s move). Recall from "How the scan works" that
the light holds still for ~3s (dwell) and then travels for ~3s (move). Don't
expect a clean on/off pattern in your plot: you should see a **noisy, jittery
line that drifts gradually up and down**, and the dwell and move segments look
much alike by eye. The 6-second timing is still there, but it's built into
how we slice up the data in Section 4, not something you can easily see on the
plot.

The first line (picking out the first 30 seconds) is done for you. Fill in the
two blanks (`____`) in the plotting cell with the column names to put on the
x-axis and the y-axis.

<details>
<summary>Stuck? Click for a hint</summary>

The column names are the ones in the table printed by the cell above. Time (in seconds) goes on the x-axis and voltage goes on the y-axis. Remember to put the names in quotes.

</details>

In [ ]:
# First, take a look at the parsed data itself (run this as-is).
# Each row is one reading: a timestamp and the panel's output voltage.
print(f"df_clean has {df_clean.shape[0]:,} rows and {df_clean.shape[1]} columns")
print(f"Spacing between samples: {df_clean['time_ms'].diff().median():.0f} ms")
print()

print("First 10 rows:")
display(df_clean.head(10))

print("Summary of the voltage column:")
display(df_clean["volts"].describe())

In [ ]:
# Pick out the first 30 seconds (done for you)
subset = df_clean[df_clean["time_s"] <= 30]

# Your turn: replace each ____ with a column name (in quotes) from the table above
plt.figure(figsize=(10, 4))
plt.plot(subset["____"], subset["____"])   # x-axis first, then y-axis
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("First 30 seconds of the clean panel trace")
plt.show()

## 4. Slice into dwell windows and average

Each scan cycle is `DWELL_S + MOVE_S` seconds long. For each of the 242 points,
we take the dwell portion, trim off the first `TRIM_S` seconds (settling time),
and average what's left. (The code that does the slicing and averaging is in `solar_helpers.py`; the cell below just runs it.) If a recording stops a little early during the very
last window, that window is still used (as long as enough samples are
available) with a warning printed -- rather than failing outright.

**Comprehension check (6 points):** looking at the trace you just plotted --
can you tell where one dwell ends and the next move begins just by looking?
Describe what you *do* see in the signal. Then explain (a) why we trim
`TRIM_S = 0.5` seconds off the **start** of each dwell window, and (b) why we
average all the remaining readings in the window instead of using just one
reading.

**Your answer:**

> _(replace this)_

In [ ]:
n_points = N_X * N_Y

values_clean = extract_point_values(
    df_clean, sample_time_clean, n_points=n_points,
    dwell_s=DWELL_S, move_s=MOVE_S, trim_s=TRIM_S,
    min_window_frac=MIN_WINDOW_FRAC, label="clean",
)
values_dirty = extract_point_values(
    df_dirty, sample_time_dirty, n_points=n_points,
    dwell_s=DWELL_S, move_s=MOVE_S, trim_s=TRIM_S,
    min_window_frac=MIN_WINDOW_FRAC, label="dirty",
)

### 4a. Exercise: quick descriptive stats

Before mapping anything spatially, get comfortable with the two arrays you
just built (`values_clean`, `values_dirty` -- each length 242, one averaged
voltage per scanned point, still in scan order, not yet a grid).

**Code exercise (3 points):** compute and print, for each array: **mean,
standard deviation, min, and max**. The clean mean is done for you as an
example. Fill in each blank (`____`) with one of these NumPy functions:
`np.mean`, `np.std`, `np.min`, `np.max`.

<details>
<summary>Stuck? Click for a hint</summary>

Look at the label printed in each line (like "Clean std"). Each blank is just the *name* of the function that matches that label -- the array it works on is already written for you, just like in the worked example.

</details>

In [ ]:
# Worked example (done for you): the mean voltage of the clean panel
print("Clean mean:", np.mean(values_clean))

# Your turn: replace each ____ with np.mean, np.std, np.min, or np.max
print("Clean std: ", ____(values_clean))
print("Clean min: ", ____(values_clean))
print("Clean max: ", ____(values_clean))
print()
print("Dirty mean:", ____(values_dirty))
print("Dirty std: ", ____(values_dirty))
print("Dirty min: ", ____(values_dirty))
print("Dirty max: ", ____(values_dirty))

**Interpretation (6 points):** Based on these numbers alone (before
seeing any map), roughly how much did the panel lose overall from soiling?
Does a lower *mean* necessarily mean *every* point on the panel got dimmer?
Why or why not -- what would you need to check to be sure?

**Your answer:**

> _(replace this)_

## 5. Reshape into grids

Each raw `values_*` array is in **scan order**, not left-to-right reading
order, because the scan zig-zags. We need to reshape each into an
`(N_Y x N_X)` grid where `grid[row, col]` = value at row `row` (0 = top),
column `col` (0 = left edge) -- regardless of which physical direction that
row was actually scanned in.

`values.reshape(N_Y, N_X)` gets you *most* of the way there: it correctly
splits the 242 values into 22 rows of 11, in the order visited. The problem
is that alternating rows were scanned in *opposite directions*, so alternating
rows of that reshaped array are backwards. You'll write that fix in the cell below.

### Build the grid

**Code exercise (15 points):** finish `reshape_to_grid()` below. For each row it has to decide whether that row was scanned right-to-left, and if so, flip it so that column 0 always means the **left edge** of the panel.

Row 0 starts at the **top-right** (`START_TOP_RIGHT = True`), so row 0 was scanned right-to-left, row 1 left-to-right, row 2 right-to-left, and so on, alternating. (If `START_TOP_RIGHT` were `False`, it would be the other way around.)

There are three blanks (`____`) to fill in:
1. With a top-right start, which value of `row % 2` (0 or 1) means "scanned right-to-left"?
2. With any other start, which value of `row % 2` means right-to-left?
3. How to reverse a row.

Run the self-check cell underneath when you're done.

<details>
<summary>Stuck? Click for a hint</summary>

- `row % 2` is the remainder when you divide the row number by 2: it's `0` for even rows (0, 2, 4, ...) and `1` for odd rows (1, 3, 5, ...).
- Python reverses a sequence with the slice `[::-1]`. For example, `[1, 2, 3][::-1]` gives `[3, 2, 1]`.

</details>

In [ ]:
def reshape_to_grid(values, n_x, n_y, start_top_right=True):
    """Turn a serpentine-scanned 1D array into a 2D grid.

    grid[row, col]: row 0 = top of panel, col 0 = LEFT edge of panel,
    regardless of which physical direction each row was actually scanned in.
    """
    values = np.asarray(values)
    if len(values) != n_x * n_y:
        raise ValueError(f"Expected {n_x*n_y} values, got {len(values)}")

    # Split the 242 values into 22 rows of 11 (in the order they were visited)
    grid = values.reshape(n_y, n_x).copy()

    for row in range(n_y):
        # Part 1: was this row scanned right-to-left?
        if start_top_right:
            right_to_left = (row % 2 == ____)
        else:
            right_to_left = (row % 2 == ____)

        # Part 2: if so, flip the row so column 0 is the LEFT edge
        if right_to_left:
            grid[row] = grid[row, ____]

    return grid

In [ ]:
# Self-check on a tiny example: 2 rows of 3 values, visited in the order 0, 1, 2, 3, 4, 5.
# With a top-right start, row 0 gets flipped and row 1 does not.
test = reshape_to_grid(np.arange(6), n_x=3, n_y=2, start_top_right=True)
print(test)
print("Top-right start correct:", np.array_equal(test, [[2, 1, 0], [3, 4, 5]]))

# With the opposite start, it's the other way around.
test2 = reshape_to_grid(np.arange(6), n_x=3, n_y=2, start_top_right=False)
print("Other start correct:    ", np.array_equal(test2, [[0, 1, 2], [5, 4, 3]]))

In [ ]:
grid_clean = reshape_to_grid(values_clean, N_X, N_Y, start_top_right=START_TOP_RIGHT)
grid_dirty = reshape_to_grid(values_dirty, N_X, N_Y, start_top_right=START_TOP_RIGHT)
grid_clean.shape, grid_dirty.shape

### 5a. Sanity-check your reshape

If your `reshape_to_grid` is correct, these two checks should both print
`True`. Row 0 (top) was scanned right-to-left, so the *first* value visited
in that row (`values_clean[0]`) should end up at the **rightmost** column of
row 0. Row 1 was scanned left-to-right, so the first value visited in that
row (`values_clean[N_X]`, i.e. point index 11) should end up at the
**leftmost** column of row 1.

In [ ]:
print("Row 0 check (top-right start):", grid_clean[0, N_X - 1] == values_clean[0])
print("Row 1 check (left-to-right):    ", grid_clean[1, 0] == values_clean[N_X])

## 6. Percent-difference grid

`(clean - dirty) / clean x 100` at each point. **Positive %** means the dirty
panel produced *less* voltage than the clean panel at that spot (i.e. a loss
due to dirt). Negative % would mean the "dirty" reading was actually higher
at that spot (possible from noise/measurement variation -- worth a second
look if it happens at many points).

**Code exercise (4 points):** compute `percent_diff_grid` using the formula
above, then print its min, max, and mean. A worked example for a *single* point
is done for you; your job is to do the same thing for the *whole grid at once*
(no loop needed -- NumPy does the math on every point together).

<details>
<summary>Stuck? Click for a hint</summary>

Copy the worked example, but replace the single-point pieces (like `grid_clean[0, 0]`) with the whole grids, `grid_clean` and `grid_dirty`. For the printing, use the same NumPy functions you used in 4a.

</details>

In [ ]:
# Worked example (done for you): percent difference at ONE point, the top-left corner
one_point = (grid_clean[0, 0] - grid_dirty[0, 0]) / grid_clean[0, 0] * 100
print(f"Top-left corner: {one_point:.1f}% loss")

# Your turn: the same formula for the WHOLE grid at once
percent_diff_grid = (____ - ____) / ____ * 100

# Then print the min, max, and mean (replace each ____ with np.min, np.max, or np.mean)
print("Min: ", ____(percent_diff_grid))
print("Max: ", ____(percent_diff_grid))
print("Mean:", ____(percent_diff_grid))

## 7. Contour plotting function

A contour map connects points of similar value into bands, the same way a
topographic map connects points of equal elevation. That makes it a natural
fit here: instead of squinting at a table of 242 numbers, you get a picture
where regions of high/low output (or high/low loss) visually pop out.

One shared function handles all three maps below. The code for it lives in `solar_helpers.py` (already imported in Setup), so you don't need
to read it. A few design choices worth noticing:

- **`smooth` / `upsample_factor`**: our real grid is only 11x22 points. That's
  too coarse to draw a smooth-looking contour directly, so we use cubic-spline
  interpolation (`scipy.ndimage.zoom`) to fill in a finer grid *for display
  only* -- the underlying data is still only 242 real measurements. `show_points=True`
  overlays the actual measurement locations so you can always see what's real
  data vs. interpolation.
- **`cmap` (colormap)**: for the clean/dirty voltage maps we use a plain
  sequential colormap (`viridis`) where "more" is just "more." For the
  percent-difference map we use a **diverging** colormap (`RdYlGn_r`) instead.
- **`diverging=True` / `TwoSlopeNorm`**: a diverging colormap needs a
  meaningful *center* value, and 0% (no change) is that center here. Without
  this, a plain colormap would visually center on whatever the min/max of the
  data happen to be, which could make a modest loss look identical to "no
  change." `TwoSlopeNorm(vcenter=0, ...)` forces green/red to mean gain/loss
  consistently, and makes the two colors symmetric around 0 regardless of
  whether the gains or the losses are larger in magnitude.

## 8. Map 1: Clean panel

Run this one as-is -- it's your template for the next two maps.

In [ ]:
fig, ax = plot_panel_contour(
    grid_clean, PANEL_WIDTH_CM, PANEL_LENGTH_CM,
    n_levels=15, cmap="viridis", title="Clean Panel Voltage Map",
    cbar_label="Volts", smooth=True, upsample_factor=8, show_points=True,
    save_path=f"{OUTPUT_DIR}/panel_contour_clean.png",
)

**Interpretation (8 points):** Describe the overall pattern in one or two
sentences -- is output roughly uniform, or is there a clear high/low region?
Then, since this is the *clean* panel, propose a plausible explanation for
why output isn't perfectly uniform even here (think about the measurement
setup, not just the panel itself).

**Your answer:**

> _(replace this)_

## 9. Map 2: Dirty panel

**Code exercise (5 points):** call `plot_panel_contour()` yourself, following
the pattern from Map 1 above. Use `grid_dirty`, title it `"Dirty Panel
Voltage Map"`, and save it to `f"{OUTPUT_DIR}/panel_contour_dirty.png"`. Keep
the other arguments the same as Map 1 so the two maps are visually
comparable (same colormap, same color scale approach).

The call is set up for you with the same arguments as Map 1. Fill in the three
blanks (`____`): which grid to plot, the title, and the file name.

<details>
<summary>Stuck? Click for a hint</summary>

Compare with the Map 1 cell: only three things change. The grid is the dirty one instead of the clean one, the title says "Dirty", and the saved file name ends in `_dirty`. Keep the quotes around the title and the `.png` in the file name.

</details>

In [ ]:
# Fill in the three blanks (____), copying the pattern from Map 1
fig, ax = plot_panel_contour(
    ____, PANEL_WIDTH_CM, PANEL_LENGTH_CM,
    n_levels=15, cmap="viridis", title="____",
    cbar_label="Volts", smooth=True, upsample_factor=8, show_points=True,
    save_path=f"{OUTPUT_DIR}/____.png",
)

**Interpretation (12 points):** (a) Where on the panel did output drop
the most -- describe the location (e.g. "bottom third," "upper-left corner,"
"one small spot near..."). (b) Do you see what looks like **one broad,
gradual change** across the panel, **one small localized anomaly**, both, or
neither? What real-world soiling causes might produce each kind of pattern
(e.g. dust settling under gravity vs. a single leaf or bird dropping)?

**Your answer:**

> _(replace this)_

## 10. Map 3: Percent difference

**Code exercise (5 points):** call `plot_panel_contour()` one more time, this
time on `percent_diff_grid`. Set `diverging=True` (so 0% sits at the center
of the colormap), title it `"Percent Difference: (Clean - Dirty) / Clean"`,
use `cbar_label="% difference (positive = loss from dirt)"`, and save it to
`f"{OUTPUT_DIR}/panel_contour_percent_diff.png"`.

The call is set up for you. Fill in the blanks (`____`): the grid, the title,
the colorbar label, whether the map is diverging (`True` or `False`), and the
file name. Everything you need to type is written in the paragraph above.

<details>
<summary>Stuck? Click for a hint</summary>

This is the same kind of call as Map 2, with one new argument: `diverging`. A diverging colormap has a meaningful center (here, 0% = no change), and `True` turns that on. Titles and labels are text, so they go in quotes.

</details>

In [ ]:
# Fill in the blanks (____) using the instructions above
fig, ax = plot_panel_contour(
    ____, PANEL_WIDTH_CM, PANEL_LENGTH_CM,
    n_levels=15, title="____", cbar_label="____",
    smooth=True, upsample_factor=8, show_points=True,
    diverging=____,
    save_path=f"{OUTPUT_DIR}/____.png",
)

**Interpretation (6 points):** Why is this map more useful for spotting
soiling *patterns* than comparing the clean and dirty maps side by side by
eye? Are there any spots where the percent difference is *negative* (dirty
reading higher than clean) -- if so, is that more likely a real physical
effect or measurement noise, and how would you decide?

**Your answer:**

> _(replace this)_

**Synthesis (15 points):** Your team has a limited maintenance budget: the
crew can only afford to fully clean **one quarter of the panel's surface
area** this visit. Using evidence from your three maps, write a short
recommendation memo (4-6 sentences) that:

- States roughly how much output the panel is losing overall (an efficiency
  number, e.g. "~X% below its clean output").
- Identifies the specific region you'd prioritize cleaning first, and why --
  cite the numbers backing that choice (e.g. how large an area, how severe
  the loss there is).
- Explains your reasoning for choosing that region over other options -- e.g.
  a small severe hotspot that's easy to fully clean vs. a larger area with a
  smaller loss per point.
- Estimates roughly how much of the panel's lost output you'd expect that
  targeted cleaning to recover, and states what's still being left on the
  table by not doing a full clean.

**Your answer:**

> _(replace this)_

## 11. Reflection: limitations of this method

A few closing questions about the method itself, not just today's results.

**Reflection (8 points):** (a) What would you gain and lose by scanning
at a coarser resolution (e.g. 5x10 = 50 points) instead of the 11x22 grid used
here? (b) The contour maps use interpolation (`upsample_factor=8`) to look
smooth -- what real feature of the panel could that interpolation cause you
to *miss*, and what could it cause you to *see* that isn't really there?
(c) This test used one artificial light held at fixed points, not real
sunlight moving across the sky -- give one way that could make today's map
not match how this panel actually performs outdoors.

**Your answer:**

> _(replace this)_

## Rerunning this notebook on a new scan

To analyze a different pair of scans (e.g. a follow-up run after cleaning, or
a different panel), change `CSV_PATH_CLEAN` and `CSV_PATH_DIRTY` in the
**Configuration** cell (Section 2) to point at the new files, then Restart &
Run All. As long as both files follow the same scope-export format and scan
geometry (11x22, 3s/3s, top-right start), everything downstream regenerates
automatically -- both voltage maps and the percent-difference map.